# Fine-Tuning Whisper for Nepali / Nepali-English Code-Switched ASR

Consolidated final pipeline for the major project *"Real-Time Speech Recognition and
Romanized Nepali-to-Devanagari Bilingual Subtitle Generation for Informative Content."*

**Pipeline:** load and clean the merged corpus -> resample to 16 kHz -> filter long
transcripts -> online audio augmentation -> log-Mel feature extraction -> LoRA
fine-tuning of `openai/whisper-medium` -> WER / CER / BLEU evaluation -> export.

**Configuration that produced the reported results:** Whisper-medium (769M parameters),
LoRA r=16 / alpha=32 on `q_proj` and `v_proj`, 8 epochs, lr 5e-5, fp16 mixed precision,
online augmentation at p=0.5.

## 1. Environment

In [ ]:
!pip install -q -U transformers datasets accelerate peft evaluate
!pip install -q jiwer librosa soundfile audiomentations sacrebleu

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import re
import random
from pathlib import Path
from functools import partial

import numpy as np
import pandas as pd
import torch

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## 2. Configuration

In [ ]:
class config:
    # Data
    ROOT_DIR = "/content/drive/MyDrive/merged_dataset"
    CSV_PATH = f"{ROOT_DIR}/merged_with_manual_remove_dirty.csv"

    # Model
    MODEL_ID = "openai/whisper-medium"
    LANG = "ne"
    TASK = "transcribe"

    # Preprocessing
    SAMPLING_RATE = 16_000
    MAX_LABEL_TOKENS = 448
    TEST_SIZE = 0.1
    SEED = 42

    # LoRA
    LORA_R = 16
    LORA_ALPHA = 32
    LORA_DROPOUT = 0.05
    LORA_TARGET_MODULES = ["q_proj", "v_proj"]

    # Training
    OUTPUT_DIR = "./whisper-lora-final"
    NUM_EPOCHS = 8
    LEARNING_RATE = 5e-5
    TRAIN_BATCH_SIZE = 32
    EVAL_BATCH_SIZE = 32
    WARMUP_STEPS = 100

    # Augmentation
    AUGMENT_PROB = 0.5

## 3. Load and clean the dataset

Drops rows with missing audio or transcripts, removes punctuation that is not spoken
aloud, collapses repeated whitespace, verifies every audio file exists on disk, and
resamples to 16 kHz -- Whisper's training rate, which also satisfies the Nyquist
criterion for the intelligible 0-8 kHz speech band.

In [ ]:
from datasets import Audio, Dataset

PUNCT_RE = re.compile(r"""[|।,?!;:"'()\[\]]""")
WS_RE = re.compile(r"\s+")


def clean_transcript(text):
    """Strip non-spoken punctuation and normalise whitespace."""
    text = str(text).strip()
    text = PUNCT_RE.sub("", text)
    text = WS_RE.sub(" ", text)
    return text.strip()


print("Loading dataset...")
df = pd.read_csv(config.CSV_PATH)
print(f"Total rows in CSV: {len(df)}")

df = df.dropna(subset=["audio_path", "transcript"])
df["transcript"] = df["transcript"].astype(str).str.strip()
df = df[df["transcript"].str.len() > 0]

df["transcript"] = df["transcript"].apply(clean_transcript)
df = df[df["transcript"].str.len() > 0]
print(f"After cleaning: {len(df)}")

# Absolute audio paths -- the merged corpus spans several source folders
df["audio"] = df["audio_path"].apply(
    lambda x: str(Path(config.ROOT_DIR) / str(x).replace("\\", "/"))
)

print("Verifying audio files exist...")
df = df[df["audio"].apply(os.path.exists)]
print(f"Valid samples: {len(df)}")

df = df[["audio", "transcript"]].rename(columns={"transcript": "sentence"})

dataset = Dataset.from_pandas(df, preserve_index=False)
dataset = dataset.cast_column("audio", Audio(sampling_rate=config.SAMPLING_RATE))

dataset = dataset.train_test_split(test_size=config.TEST_SIZE, seed=config.SEED)
print(f"Train: {len(dataset['train'])} | Eval: {len(dataset['test'])}")

## 4. Processor

In [ ]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(
    config.MODEL_ID,
    language=config.LANG,
    task=config.TASK,
)

### Filter over-long transcripts

Whisper's decoder is capped at 448 label tokens; longer sequences cause memory
failures and unstable batches during training.

In [ ]:
def is_not_too_long(sample):
    n_tokens = len(processor.tokenizer(sample["sentence"]).input_ids)
    return n_tokens <= config.MAX_LABEL_TOKENS


print("Filtering long transcripts...")
before_train, before_eval = len(dataset["train"]), len(dataset["test"])

dataset["train"] = dataset["train"].filter(is_not_too_long)
dataset["test"] = dataset["test"].filter(is_not_too_long)

print(f"Train: {before_train} -> {len(dataset['train'])}")
print(f"Eval:  {before_eval} -> {len(dataset['test'])}")

## 5. Audio augmentation (training split only)

Ranges are deliberately narrow: enough to simulate microphone noise, speaking-rate
variation, pitch differences and recording level, without distorting speech badly
enough to invalidate its transcript.

In [ ]:
from audiomentations import AddGaussianNoise, Compose, Gain, PitchShift, TimeStretch

audio_augmenter = Compose([
    # Microphone noise and recording imperfections
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.4),
    # Faster and slower speakers
    TimeStretch(min_rate=0.9, max_rate=1.1, p=0.3),
    # Speaker pitch variation
    PitchShift(min_semitones=-2, max_semitones=2, p=0.3),
    # Varying recording level
    Gain(min_gain_db=-6, max_gain_db=6, p=0.4),
])

print("Augmentation pipeline ready")

## 6. Feature extraction

Each waveform becomes a log-Mel spectrogram; each transcript becomes a sequence of
BPE token ids. Augmentation is applied only to the training split, and only to a
random `AUGMENT_PROB` fraction of its samples (online augmentation).

In [ ]:
def prepare(batch, is_training=False):
    audio = batch["audio"]
    audio_array = audio["array"]
    sr = audio["sampling_rate"]

    if is_training and random.random() < config.AUGMENT_PROB:
        try:
            audio_array = audio_augmenter(
                samples=audio_array.astype(np.float32),
                sample_rate=sr,
            )
        except Exception:
            # A failed augmentation must not drop the sample
            pass

    batch["input_features"] = processor.feature_extractor(
        audio_array,
        sampling_rate=sr,
    ).input_features[0]

    batch["labels"] = processor.tokenizer(
        batch["sentence"],
        truncation=True,
    ).input_ids

    return batch

In [ ]:
print("Processing training set (with augmentation)...")
train_dataset = dataset["train"].map(
    partial(prepare, is_training=True),
    remove_columns=dataset["train"].column_names,
    num_proc=1,  # audiomentations transforms are not picklable
)

print("Processing eval set (clean)...")
eval_dataset = dataset["test"].map(
    partial(prepare, is_training=False),
    remove_columns=dataset["test"].column_names,
    num_proc=2,
)

print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

## 7. Model and LoRA adapters

Base weights stay frozen; only low-rank matrices on the attention query and value
projections are trained. `print_trainable_parameters()` reports the trainable share,
which is roughly 0.61% of Whisper-medium's 768.6M parameters at r=16.

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(
    config.MODEL_ID,
    device_map="auto",
)

In [ ]:
lora_config = LoraConfig(
    r=config.LORA_R,
    lora_alpha=config.LORA_ALPHA,
    target_modules=config.LORA_TARGET_MODULES,
    lora_dropout=config.LORA_DROPOUT,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 8. Data collator

In [ ]:
from torch.nn.utils.rnn import pad_sequence


def data_collator(features):
    # Whisper log-Mel features are fixed-size, so they stack directly
    input_features = torch.tensor(np.array([f["input_features"] for f in features]))

    label_features = [torch.tensor(f["labels"]) for f in features]
    labels = pad_sequence(
        label_features,
        batch_first=True,
        padding_value=processor.tokenizer.pad_token_id,
    )

    # -100 is ignored by the cross-entropy loss
    labels[labels == processor.tokenizer.pad_token_id] = -100

    return {"input_features": input_features, "labels": labels}

## 9. Metrics

Word Error Rate and Character Error Rate, both reported as percentages.

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    return {
        "wer": 100 * wer_metric.compute(predictions=pred_str, references=label_str),
        "cer": 100 * cer_metric.compute(predictions=pred_str, references=label_str),
    }

## 10. Generation configuration

`forced_decoder_ids` must be cleared before training, or `predict_with_generate`
raises. Language and task are pinned so the decoder does not fall back to language
identification on short clips.

In [ ]:
model.config.forced_decoder_ids = None

model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.generation_config.language = config.LANG
model.generation_config.task = config.TASK

# Guards against the repetition loops greedy decoding is prone to on longer audio
model.generation_config.no_repeat_ngram_size = 3
model.generation_config.repetition_penalty = 1.3
model.generation_config.max_new_tokens = 128

print("Generation config set")

## 11. Training

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir=config.OUTPUT_DIR,

    per_device_train_batch_size=config.TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=config.EVAL_BATCH_SIZE,

    learning_rate=config.LEARNING_RATE,
    warmup_steps=config.WARMUP_STEPS,
    num_train_epochs=config.NUM_EPOCHS,

    fp16=True,

    do_eval=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    logging_strategy="steps",
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,   # augmented
    eval_dataset=eval_dataset,     # clean
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

In [ ]:
train_result = trainer.train()

In [ ]:
history = pd.DataFrame(trainer.state.log_history)
history.to_csv("training_log.csv", index=False)
history.tail(20)

## 12. Save and export

In [ ]:
model.save_pretrained(config.OUTPUT_DIR)
processor.save_pretrained(config.OUTPUT_DIR)
print(f"Saved to {config.OUTPUT_DIR}")

In [ ]:
import shutil

shutil.make_archive("whisper-lora-final", "zip", config.OUTPUT_DIR)

from google.colab import files
files.download("whisper-lora-final.zip")

## 13. Evaluation

### Final WER and CER on the held-out split

In [ ]:
metrics = trainer.evaluate(eval_dataset=eval_dataset)
print(f"WER: {metrics['eval_wer']:.2f}%")
print(f"CER: {metrics['eval_cer']:.2f}%")
metrics

### Per-epoch WER, CER and loss curves

In [ ]:
import matplotlib.pyplot as plt

evals = history[history["eval_wer"].notna()] if "eval_wer" in history else pd.DataFrame()
trains = history[history["loss"].notna()] if "loss" in history else pd.DataFrame()

if not evals.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].plot(evals["epoch"], evals["eval_wer"], marker="o")
    axes[0].set_title("WER by epoch")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("WER (%)")

    axes[1].plot(evals["epoch"], evals["eval_cer"], marker="o", color="tab:orange")
    axes[1].set_title("CER by epoch")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("CER (%)")

    axes[2].plot(evals["epoch"], evals["eval_loss"], marker="o", label="validation")
    if not trains.empty:
        axes[2].plot(trains["epoch"], trains["loss"], alpha=0.6, label="training")
    axes[2].set_title("Loss by epoch")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Loss")
    axes[2].legend()

    plt.tight_layout()
    plt.show()

### BLEU on the full held-out split

Reported in the project write-up to assess how closely the generated bilingual output
matches the reference transcription. Computed over the entire evaluation split rather
than a hand-picked sample, so the figure is defensible.

In [ ]:
import sacrebleu

model.eval()
device = next(model.parameters()).device

predictions, references = [], []
BATCH = 16

for start in range(0, len(eval_dataset), BATCH):
    chunk = eval_dataset[start:start + BATCH]
    feats = torch.tensor(np.array(chunk["input_features"])).to(device)

    with torch.no_grad():
        generated = model.generate(
            input_features=feats,
            language=config.LANG,
            task=config.TASK,
            max_new_tokens=128,
        )

    predictions.extend(processor.batch_decode(generated, skip_special_tokens=True))

    label_ids = [[t for t in seq if t != -100] for seq in chunk["labels"]]
    references.extend(processor.batch_decode(label_ids, skip_special_tokens=True))

    done = min(start + BATCH, len(eval_dataset))
    print(f"Decoded {done}/{len(eval_dataset)}", end="\r")

bleu = sacrebleu.corpus_bleu(predictions, [references])
print(f"\nCorpus BLEU: {bleu.score / 100:.4f}  ({bleu.score:.2f})")
print(bleu)

pd.DataFrame({"reference": references, "prediction": predictions}).to_csv(
    "eval_predictions.csv", index=False
)
print("Wrote eval_predictions.csv")

### Qualitative check

Reference and prediction side by side for a handful of held-out clips, to inspect how
code-switch points and Devanagari output are handled.

In [ ]:
for i in range(min(10, len(eval_dataset))):
    sample = eval_dataset[i]

    with torch.no_grad():
        generated = model.generate(
            input_features=torch.tensor(np.array([sample["input_features"]])).to(device),
            language=config.LANG,
            task=config.TASK,
            max_new_tokens=128,
        )

    pred = processor.decode(generated[0], skip_special_tokens=True)
    ref_ids = [t for t in sample["labels"] if t != -100]
    ref = processor.decode(ref_ids, skip_special_tokens=True)

    print(f"--- Sample {i + 1} ---")
    print(f"Reference:  {ref}")
    print(f"Prediction: {pred}")
    print(f"Exact match: {pred.strip() == ref.strip()}")
    print()